## Setup CURRENTLY A COPY OF THE BASELINE TRAINING; MODIFY FOR OUR NEW MODEL

In [ ]:
# sync with github so my imports are here
!git clone https://github.com/ellylai/10707-project.git
%cd 10707-project

!pip install -q transformers datasets scikit-learn

import sys
sys.path.append("/content/10707-project")

In [ ]:
# imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

# Adding specific directories to sys.path since they are not recognized as Python packages (missing __init__.py files).
# The parent directory '/content/10707-project' was already added in a previous cell.
import sys
if "/content/10707-project/datasets" not in sys.path:
    sys.path.append("/content/10707-project/datasets")
if "/content/10707-project/utils" not in sys.path:
    sys.path.append("/content/10707-project/utils")

from download_datasets import *
from training_utils import *
from create_dataset_splits import *
from get_waveform_dataset import *

## Data Choice 1: Unzipping XMAD Files from GDrive (Longer)

1. You must have a symlink (shortcut) from your own GDrive to the XMAD dataset, named `XMAD-dataset`.
2. Create a splits file that tells us which are in-domain and cross-domain splits.

In [ ]:
import os
from google.colab import drive

# Define languages and paths
drive.mount('/content/drive')

languages = ['ar.zip', 'de.zip', 'ro.zip', 'ru.zip', 'en.zip', 'zh-cn.zip', 'es.zip']
dataset_path = '/content/drive/MyDrive/XMAD-dataset'
target_dir = '/content/xmad_local'
os.makedirs(target_dir, exist_ok=True)

# Build a single bash command string
# The '&' at the end of each unzip command sends it to the background
# 'wait' ensures the cell doesn't finish until all background tasks are done
commands = []
for lang in languages:
    zip_path = os.path.join(dataset_path, lang)
    if os.path.exists(zip_path):
        # -q for quiet (hiding the thousands of file names speeds it up)
        # -n to skip existing
        cmd = f"unzip -qn '{zip_path}' -d {target_dir} &"
        commands.append(cmd)

full_cmd = "\n".join(commands) + "\nwait\necho 'All unzipping finished!'"

# Execute everything in parallel
with open('unzip_all.sh', 'w') as f:
    f.write("#!/bin/bash\n" + full_cmd)

!bash unzip_all.sh

In [ ]:
!pip install awscli
!aws s3api get-bucket-location --bucket 10707-project
!aws configure

### Create Splits File

In [ ]:
import pandas as pd
import os
from pathlib import Path

# Root directory where your languages are unzipped
root_dir = "/content/xmad_local"
all_data = []

# Walk through all directories to find meta.csv files
for path in Path(root_dir).rglob('meta.csv'):
    # Read the individual metadata file
    df = pd.read_csv(path)

    # Identify the dataset source from the folder structure
    parent_folder = path.parent.name

    if "commonvoice" in parent_folder.lower():
        # CommonVoice contains the internal 'train' and 'test' labels
        # We use 'train' for training and 'test' as our Validation set
        df['split'] = df['split'].map({'train': 'train', 'test': 'val'})
    else:
        # AISHELL-3, M-AILABS, etc., are strictly for Cross-Domain Testing
        df['split'] = 'test'

    # Convert relative 'sample_name' paths to absolute 'file' paths for the DataLoader
    # This ensures your training_pipeline.ipynb can find the .wav files
    # The original error was 'KeyError: 'file'' because the column is actually 'sample_name'
    df['file'] = df.apply(
        lambda row: os.path.join(
            path.parent,
            'fake' if row['is_fake'] == 1 else 'real',
            str(row['sample_name'])
        ),
        axis=1
    )

    # 3. Rename label for clarity in your DataLoader
    df['label'] = df['is_fake']

    all_data.append(df)

# Combine all parsed metadata into one master dataframe
if all_data:
    master_df = pd.concat(all_data, ignore_index=True)

    master_df['split'] = master_df['split'].fillna('val')

    # Save the manifest to the path expected by your notebook
    output_path = "/content/10707-project/speechfake_splits.csv"
    master_df.to_csv(output_path, index=False)

    print(f"Successfully created: {output_path}")

    # upload to s3 bucket
    s3_dest = "s3://10707-project/xmad_bench/metadata/speechfake_splits.csv"
    !aws s3 cp {output_path} {s3_dest}
    print(f"Successfully uploaded to S3: {s3_dest}")

    print("Split Distribution:")
    print(master_df['split'].value_counts())

    print("Unique values in the split column:")
    print(master_df['split'].unique())

    print("\nRows where split is NaN:")
    print(master_df['split'].isna().sum())
else:
    print("No meta.csv files found. Ensure the unzip process finished correctly.")

### Upload Files as tarball to S3 for future use

In [ ]:
import os

# 1. Config
SOURCE_DIR = "/content/xmad_local"
TAR_NAME = "xmad_bench.tar"  # Removed .gz for 10x faster extraction later
S3_DEST = f"s3://10707-project/{TAR_NAME}"

# 2. Tune AWS CLI for a single massive stream
!aws configure set default.s3.multipart_threshold 64MB
!aws configure set default.s3.multipart_chunksize 64MB
!aws configure set default.s3.max_concurrent_requests 20

# 3. Stream Tar directly to S3 (No local file created)
print(f"📦 Streaming {SOURCE_DIR} directly to {S3_DEST}...")
print("🚀 This skips local disk writes and uses your full network bandwidth.")

# -c: create, -f -: output to stdout (pipe)
!tar -cf - -C {SOURCE_DIR} . | aws s3 cp - {S3_DEST} --expected-size 150000000000

# 4. Verification
print(f"\n✅ Upload complete. Verification:")
!aws s3 ls {S3_DEST}

## Data Choice 2: Download tarball from S3 (if it's in there)

It might not be in there... take a look at the S3 bucket and see what that mess looks like...

In [ ]:
!pip install awscli
!aws configure

!aws s3 cp s3://10707-project/xmad_bench/de.tar - | tar -xf - -C /content/xmad_local
!aws s3 cp s3://10707-project/xmad_bench/es.tar - | tar -xf - -C /content/xmad_local
!aws s3 cp s3://10707-project/xmad_bench/zh-cn.tar - | tar -xf - -C /content/xmad_local
!aws s3 cp s3://10707-project/xmad_bench/ro.tar - | tar -xf - -C /content/xmad_local
!aws s3 cp s3://10707-project/xmad_bench/ru.tar - | tar -xf - -C /content/xmad_local

## Dataset & Dataloader

Create a class for the XMAD dataset and create dataloaders to feed into the model.

In [23]:
import torch
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader

class XMADDataset(Dataset):
    def __init__(self, df, target_sr=16000, max_seconds=4.0):
        self.df = df
        self.target_sr = target_sr
        self.max_samples = int(target_sr * max_seconds)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['file']
        label = row['is_fake'] # Corrected: Use 'is_fake' column directly

        # Load audio
        waveform, sr = torchaudio.load(file_path)

        # Resample if necessary
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(sr, self.target_sr)
            waveform = resampler(waveform)

        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Pad or Trim to fixed length (crucial for batching)
        if waveform.shape[1] > self.max_samples:
            waveform = waveform[:, :self.max_samples]
        else:
            padding = self.max_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))

        return waveform.squeeze(0), torch.tensor(label)

In [ ]:
# Load your master manifest
master_df = pd.read_csv("/content/10707-project/speechfake_splits.csv")

# Filter dataframes by split
train_df = master_df[master_df['split'] == 'train'].reset_index(drop=True)
val_df = master_df[master_df['split'] == 'val'].reset_index(drop=True)
test_df = master_df[master_df['split'] == 'test'].reset_index(drop=True)

def print_stratification(df, name):
    print(f"\n--- {name} Stratification ---")
    print(f"Total Samples: {len(df)}")

    for col in ['is_fake', 'label', 'meta']:
        if col in df.columns:
            print(f"\nProportions for '{col}':")
            # normalize=True gives ratios (0.0 to 1.0)
            print(df[col].value_counts(normalize=True).map(lambda n: f'{n:.2%}'))
        else:
            print(f"\nColumn '{col}' not found in this split.")

# Run the check
print_stratification(train_df, "Train")
print_stratification(val_df, "Validation")
print_stratification(test_df, "Test")

# Create Dataset objects
train_dataset = XMADDataset(train_df)
val_dataset = XMADDataset(val_df)
test_dataset = XMADDataset(test_df)

# Create DataLoaders
# Set num_workers to 2 or 4 to speed up loading on Colab
train_loader = DataLoader(train_dataset, batch_size=96, shuffle=True, num_workers=8, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=96, shuffle=False, num_workers=8, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=96, shuffle=False, num_workers=8, pin_memory=True)

print(f"Loaders created: {len(train_loader)} train batches, {len(val_loader)} val batches, {len(test_loader)} test batches.")

## Start Training

In [ ]:
# args/configs for AcousticStream
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 0.0001
epochs = 10

In [ ]:
# wandb and huggingface setup
import wandb
from huggingface_hub import HfApi, login

# Initialize W&B
wandb.init(
    project="audio-deepfake-detection",
    config={
        "learning_rate": lr,
        "architecture": "Dual_Stream",
        "dataset": "XMAD-Bench",
        "epochs": epochs,
    }
)

# Login to HF (Run this once or use a token)
login()
api = HfApi()
repo_id = "Joel-10707-Project-S26/model-acousticstream"

In [ ]:
# import model and initialize optimizer: TODO
import sys
# Ensure the model directory is in sys.path for direct module import
if "/content/10707-project/model" not in sys.path:
    sys.path.append("/content/10707-project/model")

# IMPORT THE CORRECT MODEL HERE
from acousticstream import AcousticStream

model = AcousticStream(standalone=True).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

### Test before running
1. Test if we can write to HF
2. Test if we can run a training loop

In [ ]:
import os
import huggingface_hub

# 2. Re-initialize the API object
# This forces the client to read the new environment variable
from huggingface_hub import hf_hub_download, HfApi
# api = HfApi(token=os.environ["HF_ADL_WRITE"])

# 3. Verify
try:
    user_info = api.whoami()
    print(f"Successfully switched! Logged in as: {user_info['name']}")
    print(f"Can write to Joel-10707-Project-S26: {'Yes' if any(org['name'] == 'Joel-10707-Project-S26' for org in user_info['orgs']) else 'No'}")
except Exception as e:
    print(f"Switch failed: {e}")


repo_id = "Joel-10707-Project-S26/model-acousticstream"

# 1. Create a tiny test file locally
test_file = "elly_write_test.txt"
with open(test_file, "w") as f:
    f.write("Elly has write access and is ready for 10707 training!")

# 2. Attempt to upload DIRECTLY (No create_repo call)
try:
    print(f"Testing direct write access to {repo_id}...")
    api.upload_file(
        path_or_fileobj=test_file,
        path_in_repo="tests/elly_write_test.txt",
        repo_id=repo_id,
        # Since it's a private repo, specify the repo_type if needed,
        # though it defaults to model
        repo_type="model"
    )
    print("\n✅ Success! You can push to the repo.")
except Exception as e:
    print(f"\n❌ Still failing: {e}")

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# --- 1. Define Dummy Dimensions ---
# Waveform length for W2V2 is typically 64,000 samples (4 seconds at 16kHz)
sample_rate = 16000
duration = 4
waveform_len = sample_rate * duration
batch_size = 2 # Small batch for testing

# --- 2. Create Synthetic Data ---
# Random noise waveforms and random binary labels (0 or 1)
dummy_waveforms = torch.randn(batch_size, waveform_len)
dummy_labels = torch.randint(0, 2, (batch_size,))

# Create a tiny dataset
dummy_dataset = TensorDataset(dummy_waveforms, dummy_labels)

# --- 3. Replace Loaders ---
# Temporarily overwrite your real loaders
test_train_loader = DataLoader(dummy_dataset, batch_size=batch_size)
test_val_loader = DataLoader(dummy_dataset, batch_size=batch_size)

print(f"✅ Created dummy loaders with shape: {dummy_waveforms.shape}")

# NOW REPLACE YOUR REAL LOADERS IN THE TRAINING LOOP WITH THESE DUMMY LOADERS TO TEST

### Actual Training
1. load from checkpoint if one is there
2. training loop

In [ ]:
# Resume from Checkpoint if available
repo_id = "Joel-10707-Project-S26/model-acousticstream"
local_checkpoint_dir = "/content/checkpoints"

start_epoch = 0
best_val_acc = 0.0

latest_checkpoint = "None"
start_epoch = 0

try:
    print(f"Checking for existing checkpoints in {repo_id}...")
    files = api.list_repo_files(repo_id=repo_id)
    # Find all epoch checkpoints and pick the highest number
    checkpoint_files = [f for f in files if f.startswith("checkpoint-epoch-") and f.endswith(".pt")]

    if checkpoint_files:
        # Sort by epoch number to get the latest
        latest_checkpoint = sorted(checkpoint_files, key=lambda x: int(x.split('-')[-1].split('.')[0]))[-1]
        print(f"Found latest checkpoint: {latest_checkpoint}. Downloading...")

        path = hf_hub_download(repo_id=repo_id, filename=latest_checkpoint, local_dir=local_checkpoint_dir)
        checkpoint = torch.load(path, map_location=device)

        # Load states
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] # This will be the next epoch to run
        best_val_acc = checkpoint.get('val_acc', 0.0)

        print(f"✅ Successfully resumed from Epoch {start_epoch}. Best Val Acc so far: {best_val_acc:.4f}")
    else:
        print("No checkpoints found. Starting training from scratch.")
except Exception as e:
    print(f"Note: Could not resume (likely repo is empty or connection issue). Starting fresh. Error: {e}")

In [ ]:
# OPTIONAL: CUT THE TRAINING IN HALF FOR SPEED
from torch.utils.data import Subset
import numpy as np

# 1. Generate random indices for 50% of the data
indices = np.arange(len(train_dataset))
np.random.shuffle(indices)
subset_size = int(0.5 * len(train_dataset))
train_indices = indices[:subset_size]

# 2. Create the subset
train_subset = Subset(train_dataset, train_indices)

# 3. Create a new loader for the loop
# Using your optimized BS=96 or 128 from earlier
train_loader_half = DataLoader(
    train_subset,
    batch_size=96,
    shuffle=True,
    num_workers=8,
    pin_memory=True
)

print(f"Original: {len(train_dataset)} | Sampled: {len(train_subset)}")

In [ ]:
train_speakers = set(train_df['speaker_id'])
val_speakers = set(val_df['speaker_id'])
overlap = train_speakers.intersection(val_speakers)
print(f"Number of overlapping speakers: {len(overlap)}")

In [ ]:
# train
torch.cuda.empty_cache()

for epoch in range(start_epoch, epochs):
    print(f"EPOCH: {epoch+1}")
    # --- Existing Training Logic ---
    train_loss = train_epoch(model, train_loader_half, optimizer, criterion, device)
    torch.cuda.empty_cache()

    val_loss, val_acc = validate(model, val_loader, criterion, device)

    wandb.log({
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "val/loss": val_loss,
        "val/accuracy": val_acc
    })

    print(f"Epoch {epoch+1}: Loss {train_loss:.4f}, Vall Loss {val_loss:.4f}, Val Acc {val_acc:.4f}")

    # --- Checkpointing & HF Upload ---
    checkpoint_name = f"checkpoint-epoch-{epoch+1}.pt"
    checkpoint_path = os.path.join("/content/", checkpoint_name)

    # Save locally first
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
        'val_acc': val_acc,
    }, checkpoint_path)

    try:
        print(f"Uploading {checkpoint_name}...")
        api.upload_file(
            path_or_fileobj=checkpoint_path,
            path_in_repo=checkpoint_name,
            repo_id=repo_id,
            commit_message=f"Epoch {epoch+1} - Acc: {val_acc:.4f}"
        )
        os.remove(checkpoint_path)
    except Exception as e:
        print(f"HF Upload failed, but local checkpoint saved: {e}")

    torch.cuda.empty_cache()

### Test on Unseen Cross-Domain Data

In [ ]:
# test
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def calculate_eer(y_true, y_scores):
    # y_scores should be the probability of the "Fake" class
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr

    # The EER is where fpr == fnr
    eer = fpr[np.nanargmin(np.absolute((fnr - fpr)))]
    return eer

def evaluate_model(model, test_loader, device, criterion=None):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    total_loss = 0.0

    print("🔍 Running Final Evaluation on Test Set...")

    with torch.no_grad():
        for waveforms, labels in tqdm(test_loader):
            waveforms, labels = waveforms.to(device), labels.to(device)

            # Forward pass
            hidden_layer, outputs = model(waveforms)

            if criterion:
                loss = criterion(outputs, labels)
                total_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)[:, 1]

            # Get predictions (assuming CrossEntropy / Multi-class)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    # --- Metrics Calculation ---
    avg_loss = total_loss / len(test_loader) if criterion else "N/A"
    acc = accuracy_score(all_labels, all_preds)
    eer_val = calculate_eer(all_labels, all_probs)

    print("\n" + "="*30)
    print(f"TEST RESULTS")
    print(f"Average Loss: {avg_loss}")
    print(f"Accuracy:     {acc:.4f}")
    print(f"Final EER: {eer_val * 100:.2f}%")
    print("="*30)

    # Detailed Report (Precision, Recall, F1)
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['Real', 'Fake']))

    # --- Visualization: Confusion Matrix ---
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix - XMAD Test Set')
    plt.show()

    return all_labels, all_preds

# --- Execute Test ---
labels, predictions = evaluate_model(model, test_loader, device, criterion)